In [3]:
label_map = {
    "normal_ecg_images": 0,
    "abnormal_heartbeat_ecg_images": 1,
    "myocardial_infarction_ecg_images": 2,
    "post_mi_history_ecg_images": 3
}

In [4]:
import os
import pandas as pd

dataset_path = "../datasets/ecg_data"

data = []

for folder_name, label in label_map.items():

    folder_path = os.path.join(dataset_path, folder_name)

    for image_name in os.listdir(folder_path):

        image_path = os.path.join(folder_path, image_name)

        data.append([image_path, label])

df = pd.DataFrame(
    data,
    columns=["image_path", "label"]
)
# here we combined all the images and label them according to the classes
df.head()

,image_path,label
0,../datasets/ecg_data/normal_ecg_images/Normal(...,0
1,../datasets/ecg_data/normal_ecg_images/Normal(...,0
2,../datasets/ecg_data/normal_ecg_images/Normal(...,0
3,../datasets/ecg_data/normal_ecg_images/Normal(...,0
4,../datasets/ecg_data/normal_ecg_images/Normal(...,0


In [5]:
print(df.shape)

(928, 2)


In [6]:
print(df["label"].value_counts())

label
0    284
2    239
1    233
3    172
Name: count, dtype: int64


In [7]:
 # Now we dive the data set into two parts one is form training and other is form testing for this we use a function the is train_test_split function in sklearn

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [8]:
print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (742, 2)
Test : (186, 2)


In [9]:
# this time to load our VIT model and train it

from transformers import ViTImageProcessor, ViTForImageClassification

processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=4,
    ignore_mismatched_sizes=True
)

print("ViT Loaded Successfully")

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT Loaded Successfully


In [10]:
from torch.utils.data import Dataset       # here use torch dataset template for handling the data
from PIL import Image                #use to open images

class ECGDataset(Dataset):

    def __init__(self, dataframe, processor):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor           # store the vit image processor and use to convert the images into vit input

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = self.dataframe.iloc[idx]["image_path"]
        label = self.dataframe.iloc[idx]["label"]

        image = Image.open(image_path).convert("RGB")

        encoding = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": label
        }

In [11]:
train_dataset = ECGDataset(train_df, processor)
test_dataset = ECGDataset(test_df, processor)

print(len(train_dataset))
print(len(test_dataset))

742
186


In [12]:
sample = train_dataset[0]

print(sample["pixel_values"].shape)
print(sample["labels"])

torch.Size([3, 224, 224])
2


In [13]:
# creating dataloaders

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 93
Test batches: 24


In [14]:
# Moving model to gpu 

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print(device)

cuda


In [15]:
# Define optimizer

from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Optimizer Ready")

Optimizer Ready


In [16]:
# quick training test 

batch = next(iter(train_loader))

print(batch["pixel_values"].shape)
print(batch["labels"].shape)

torch.Size([8, 3, 224, 224])
torch.Size([8])


In [17]:
# Training lopp

import torch
from tqdm import tqdm

epochs = 3    #The model sees all 742 training images 3 times

for epoch in range(epochs):

    model.train()

    total_loss = 0

    progress_bar = tqdm(train_loader)   #creat a visual progress bar

    for batch in progress_bar:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_description(
            f"Epoch {epoch+1}/{epochs}"
        )

        progress_bar.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Loss: {avg_loss:.4f}")

Epoch 1/3: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 93/93 [01:02<00:00,  1.49it/s, loss=0.231]



Epoch 1 Loss: 0.6726


Epoch 2/3: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 93/93 [00:57<00:00,  1.61it/s, loss=0.0063]



Epoch 2 Loss: 0.1704


Epoch 3/3: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 93/93 [01:00<00:00,  1.55it/s, loss=0.0248]


Epoch 3 Loss: 0.0661


In [18]:
# Evaluation

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values)

        predictions = torch.argmax(outputs.logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.967741935483871


In [19]:
# Classification report

from sklearn.metrics import classification_report

y_true = []
y_pred = []

model.eval()

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values)

        predictions = torch.argmax(outputs.logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97        57
           1       1.00      0.91      0.96        47
           2       0.94      1.00      0.97        48
           3       1.00      0.94      0.97        34

    accuracy                           0.97       186
   macro avg       0.97      0.96      0.97       186
weighted avg       0.97      0.97      0.97       186



In [20]:
# Saving the model
model.save_pretrained("../trained_models/vit_ecg_model")
processor.save_pretrained("../trained_models/vit_ecg_model")

print("Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully


In [21]:
from PIL import Image
import torch
from transformers import ViTImageProcessor, ViTForImageClassification

model_path = "../trained_models/vit_ecg_model"

processor = ViTImageProcessor.from_pretrained(model_path)
model = ViTForImageClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [22]:
# creating prediction function

label_map = {
    0: "Normal ECG",
    1: "Abnormal Heartbeat",
    2: "Myocardial Infarction",
    3: "Post MI History"
}

def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax(-1).item()

    return label_map[predicted_class]

In [23]:
# predict_image("/home/avanash/Codding Section/Machine Learning/From Internship/MINI Projects/Heart Disease AI/datasets/ecg_data/normal_ecg_images/Normal(1).jpg")

'Normal ECG'